# 01 - Data exploration

Explores the SIDTD and FantasyID datasets: folder structure, file types, image sizes, sample images,
official splits, forgery metadata and split leakage.

> **Clear all outputs before committing.** FantasyID contains real faces.

In [ ]:
from pathlib import Path
from collections import Counter
import json
import random
import re

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

ROOT = Path.cwd()
while not (ROOT / ".gitignore").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

SIDTD = ROOT / "data" / "sidtd"
FANTASY = ROOT / "data" / "fantasyid" / "FantasyID"
IMG_EXT = {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp"}

print(ROOT)
print("SIDTD:", SIDTD.exists(), "| FantasyID:", FANTASY.exists())

## 1. Folder structure

In [ ]:
def folder_summary(root, max_depth=4):
    rows = []
    for folder in sorted(p for p in root.rglob("*") if p.is_dir()):
        depth = len(folder.relative_to(root).parts)
        if depth > max_depth:
            continue
        n_images = sum(1 for f in folder.iterdir() if f.suffix.lower() in IMG_EXT)
        rows.append({"folder": str(folder.relative_to(root)), "images": n_images})
    return pd.DataFrame(rows)

pd.set_option("display.max_rows", 200)
display(folder_summary(SIDTD))
display(folder_summary(FANTASY))

## 2. File types

In [ ]:
for name, root in [("SIDTD", SIDTD), ("FantasyID", FANTASY)]:
    counts = Counter(f.suffix.lower() for f in root.rglob("*") if f.is_file())
    print(name, counts.most_common())

## 3. Image sizes

In [ ]:
def size_summary(root, n=200, seed=0):
    files = [f for f in root.rglob("*") if f.suffix.lower() in IMG_EXT]
    random.Random(seed).shuffle(files)
    return Counter(Image.open(f).size for f in files[:n]).most_common(10)

print("SIDTD:", size_summary(SIDTD))
print("FantasyID:", size_summary(FANTASY))

## 4. Sample images

In [ ]:
def show_samples(root, n=8, seed=0):
    files = [f for f in root.rglob("*") if f.suffix.lower() in IMG_EXT]
    random.Random(seed).shuffle(files)
    fig, axes = plt.subplots(2, n // 2, figsize=(16, 7))
    for ax, f in zip(axes.ravel(), files[:n]):
        ax.imshow(Image.open(f).convert("RGB"))
        ax.set_title(str(f.relative_to(root))[-45:], fontsize=7)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

show_samples(SIDTD)
show_samples(FANTASY)

## 5. SIDTD official splits

In [ ]:
splits = {s: pd.read_csv(SIDTD / "split_normal" / f"{s}_split_SIDTD.csv") for s in ["train", "val", "test"]}
split_counts = pd.concat(
    {s: d.groupby(["class_name", "label_name"]).size().unstack(fill_value=0) for s, d in splits.items()},
    axis=1,
)
display(split_counts)

## 6. SIDTD forgery metadata

In [ ]:
def print_json(path, n_chars=2000):
    data = json.load(open(path, encoding="utf-8"))
    print("==", path.name, "==")
    print(json.dumps(data, ensure_ascii=False, indent=1)[:n_chars], "\n")

fakes = sorted((SIDTD / "templates/Images/fakes").glob("*.jpg"))
print([f.name for f in fakes[:8]])
print(Counter(re.sub(r"\d+", "N", f.stem) for f in fakes).most_common(10))

real_ann = sorted((SIDTD / "templates/Annotations/reals").glob("*.json"))
fake_ann = sorted((SIDTD / "templates/Annotations/fakes").glob("*.json"))
print("real JSON:", len(real_ann), "| fake JSON:", len(fake_ann))
print_json(fake_ann[0])
print_json(real_ann[0], n_chars=1500)

In [ ]:
fake_meta = pd.DataFrame([json.load(open(p, encoding="utf-8")) for p in fake_ann])
fake_meta["class_name"] = fake_meta["name"].str.split("_").str[0]
display(fake_meta["ctype"].value_counts())
display(pd.crosstab(fake_meta["class_name"], fake_meta["ctype"]))
display(pd.crosstab(fake_meta["field"], fake_meta["ctype"]))
print("Crop & Replace with a second source:",
      (fake_meta.loc[fake_meta.ctype == "Crop_and_Replace", "second_src"] != "None").sum())

## 7. Leakage and missing files in the official split

In [ ]:
def source_doc(path):
    return Path(path).stem.split("_fake_")[0]

for s, d in splits.items():
    d["source"] = d["image_path"].map(source_doc)
    d["exists"] = d["image_path"].map(lambda p: (SIDTD / p).exists())
    print(f"{s}: {len(d)} rows, {d['image_path'].nunique()} unique, missing files = {(~d['exists']).sum()}")

print("train-test shared images:", len(set(splits["train"].image_path) & set(splits["test"].image_path)))
print("train-test shared source documents:", len(set(splits["train"].source) & set(splits["test"].source)))

## 8. FantasyID structure

In [ ]:
rows = []
for f in FANTASY.rglob("*.jpg"):
    parts = f.relative_to(FANTASY).parts
    if parts[0] == "examples":
        continue
    rows.append({
        "split": parts[0],
        "label": parts[1],
        "attack": parts[2] if len(parts) == 5 else "-",
        "device": parts[-2],
        "template": f.stem.split("-")[0],
    })
fantasy = pd.DataFrame(rows)
display(fantasy.groupby(["split", "label", "attack"]).size())
display(fantasy[fantasy["template"].str.startswith("turk")].groupby(["split", "label", "attack"]).size())
display(fantasy.groupby(["template", "split"]).size().unstack(fill_value=0))

In [ ]:
for md_file in FANTASY.rglob("*.md"):
    print(md_file.read_text(encoding="utf-8")[:3000])

altered = Counter()
for p in FANTASY.glob("**/attack/**/*.json"):
    for region in json.load(open(p, encoding="utf-8")).get("regions", []):
        attrs = region["region_attributes"]
        if attrs.get("region_provenance") == "altered":
            altered[attrs["field_name"]] += 1
print(altered.most_common())